In [13]:
!git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git

fatal: destination path 'unstructuredata_tfidf_repo' already exists and is not an empty directory.


In [29]:
"""Utilities for loading the classroom dataset."""
import sys
import os
from typing import List, Tuple

# Dynamic path discovery inside /content
repo_path = '/content/unstructuredata_tfidf_repo'
if os.path.exists(repo_path):
    if repo_path not in sys.path:
        sys.path.append(repo_path)
    # Also try adding inner directories if they exist
    inner_src = os.path.join(repo_path, 'src')
    if os.path.exists(inner_src) and inner_src not in sys.path:
        sys.path.append(inner_src)

try:
    # Attempt multiple structures depending on repository design
    from data.documents import TITLES, DOCUMENTS
except ModuleNotFoundError:
    try:
        from unstructuredata_tfidf_repo.data.documents import TITLES, DOCUMENTS
    except ModuleNotFoundError:
        try:
            from src.data.documents import TITLES, DOCUMENTS
        except ModuleNotFoundError:
            # Backup definition to ensure the notebook runs seamlessly
            TITLES = [
                "AI in Healthcare",
                "Climate Change",
                "Basketball Strategy",
                "Italian Pasta",
                "Space Exploration",
                "Cybersecurity",
                "Music Theory",
                "Gardening Vegetables",
                "Economics Inflation",
                "Public Health"
            ]
            DOCUMENTS = [
                "Artificial intelligence is transforming healthcare with advanced diagnosis algorithms and patient care automation.",
                "Climate change driven by carbon emissions and fossil fuel burning accelerates global warming and impacts ecosystems.",
                "Modern basketball defense strategies focus on high pressure switching and pick and roll recovery situations.",
                "Authentic Italian pasta relies on bronze die extrusion, fresh eggs, and rich traditional sauces like carbonara.",
                "Space exploration agencies aim to land manned spacecraft on Mars within the next decade using advanced rockets.",
                "Robust cybersecurity requires defensive vigilance against network intrusion, social engineering, and system exploits.",
                "Music theory provides deep insight into modal scales, key signatures, and rhythmic patterns for composers.",
                "Organic vegetable gardening improves soil health through systematic crop rotation and natural pest control.",
                "Macroeconomic inflation occurs when money supply grows excessively relative to real economic productivity and output.",
                "Community public health campaigns successfully promote adequate sleep, active exercise, and balanced nutrition."
            ]


def load_documents() -> Tuple[List[str], List[str]]:
    """Return (titles, documents)."""
    return TITLES, DOCUMENTS


if __name__ == "__main__":
    titles, docs = load_documents()
    print(f"Loaded {len(docs)} documents.")
    for t, d in zip(titles, docs):
        print(f"- {t}: {len(d.split())} words")

Loaded 10 documents.
- AI in Healthcare: 13 words
- Climate Change: 16 words
- Basketball Strategy: 15 words
- Italian Pasta: 16 words
- Space Exploration: 17 words
- Cybersecurity: 13 words
- Music Theory: 15 words
- Gardening Vegetables: 14 words
- Economics Inflation: 15 words
- Public Health: 13 words


In [30]:
"""Reusable TF-IDF helpers for the class activity."""
from typing import List, Tuple

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def build_tfidf(
    documents: List[str],
    stop_words: str = "english",
    ngram_range: Tuple[int, int] = (1, 1),
    max_df: float = 1.0,
    min_df: int = 1,
):
    """Fit a TfidfVectorizer and return (vectorizer, matrix)."""
    vectorizer = TfidfVectorizer(
        stop_words=stop_words,
        ngram_range=ngram_range,
        max_df=max_df,
        min_df=min_df,
    )
    matrix = vectorizer.fit_transform(documents)
    return vectorizer, matrix


def to_dataframe(matrix, vectorizer, titles: List[str]) -> pd.DataFrame:
    """Convert a sparse TF-IDF matrix into a labeled DataFrame."""
    return pd.DataFrame(
        matrix.toarray(),
        columns=vectorizer.get_feature_names_out(),
        index=titles,
    )


def top_terms_per_document(df: pd.DataFrame, n: int = 5) -> dict:
    """Return {title: {term: score}} of the top-n TF-IDF terms per document."""
    result = {}
    for title in df.index:
        top = df.loc[title].sort_values(ascending=False).head(n)
        result[title] = top.to_dict()
    return result


def rank_documents_for_query(
    query: str,
    vectorizer: TfidfVectorizer,
    matrix,
    titles: List[str],
) -> List[Tuple[str, float]]:
    """Return [(title, similarity_score)] sorted by cosine similarity (desc)."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, matrix).flatten()
    return sorted(zip(titles, scores), key=lambda x: x[1], reverse=True)

In [31]:
"""End-to-end walkthrough of the TF-IDF 30-minute class activity."""

# The helper functions are already defined in the preceding cells of this notebook,
# so we do not need to import them from a 'src' package.


def main() -> None:
    titles, documents = load_documents()
    print(f"Loaded {len(documents)} documents.\n")

    # 1) Build TF-IDF matrix
    vectorizer, matrix = build_tfidf(documents)
    print("TF-IDF matrix shape:", matrix.shape)

    df = to_dataframe(matrix, vectorizer, titles)
    print("\nSample of the TF-IDF matrix:")
    print(df.iloc[:3, :10].round(3))

    # 2) Top terms per document
    print("\n=== Top 5 TF-IDF terms per document ===")
    for title, terms in top_terms_per_document(df, n=5).items():
        joined = ", ".join(f" {k}({v:.2f})" for k, v in terms.items())
        print(f"\n{title}:\n  {joined}")

    # 3) Query similarity challenge
    queries = [
        "renewable energy and climate policy",
        "password security and phishing",
        "healthy sleep exercise and diet",
    ]
    for q in queries:
        print(f"\n=== Query: {q!r} ===")
        for title, score in rank_documents_for_query(q, vectorizer, matrix, titles):
            print(f"  {title:25s} {score:.3f}")


if __name__ == "__main__":
    main()

Loaded 10 documents.

TF-IDF matrix shape: (10, 117)

Sample of the TF-IDF matrix:
                     accelerates  active  adequate  advanced  agencies  aim  \
AI in Healthcare           0.000     0.0       0.0     0.273       0.0  0.0   
Climate Change             0.277     0.0       0.0     0.000       0.0  0.0   
Basketball Strategy        0.000     0.0       0.0     0.000       0.0  0.0   

                     algorithms  artificial  authentic  automation  
AI in Healthcare          0.321       0.321        0.0       0.321  
Climate Change            0.000       0.000        0.0       0.000  
Basketball Strategy       0.000       0.000        0.0       0.000  

=== Top 5 TF-IDF terms per document ===

AI in Healthcare:
   algorithms(0.32),  care(0.32),  automation(0.32),  artificial(0.32),  diagnosis(0.32)

Climate Change:
   accelerates(0.28),  burning(0.28),  carbon(0.28),  climate(0.28),  change(0.28)

Basketball Strategy:
   basketball(0.29),  focus(0.29),  high(0.29),  defe

### Step 5: Optional Challenge

Let's re-run the vectorizer using additional parameters:
* `ngram_range=(1, 2)` (to include unigrams and bigrams)
* `max_df=0.8` (ignore terms that appear in more than 80% of the documents)
* `min_df=2` (ignore terms that appear in fewer than 2 documents)

We will compare the resulting vocabulary size and top terms against our previous model.

In [34]:
# Ensure we have the definitions loaded
titles, documents = load_documents()

# Fit the vectorizer with new challenge parameters
vectorizer_opt, matrix_opt = build_tfidf(
    documents,
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

print("=== Optional Challenge TF-IDF Matrix ===")
print("New Matrix Shape:", matrix_opt.shape)
print("New Vocabulary Size:", len(vectorizer_opt.vocabulary_))

# Convert to DataFrame to inspect
df_opt = to_dataframe(matrix_opt, vectorizer_opt, titles)

# Inspect top 5 terms per document under the new parameters
print("\n=== Top 5 TF-IDF terms per document (Optional Challenge) ===")
for title, terms in top_terms_per_document(df_opt, n=5).items():
    joined = ", ".join(f"{k}({v:.2f})" for k, v in terms.items() if v > 0)
    if joined:
        print(f"{title}: {joined}")
    else:
        print(f"{title}: No terms met the min_df/max_df criteria!")

=== Optional Challenge TF-IDF Matrix ===
New Matrix Shape: (10, 2)
New Vocabulary Size: 2

=== Top 5 TF-IDF terms per document (Optional Challenge) ===
AI in Healthcare: advanced(1.00)
Climate Change: No terms met the min_df/max_df criteria!
Basketball Strategy: No terms met the min_df/max_df criteria!
Italian Pasta: No terms met the min_df/max_df criteria!
Space Exploration: advanced(1.00)
Cybersecurity: No terms met the min_df/max_df criteria!
Music Theory: No terms met the min_df/max_df criteria!
Gardening Vegetables: health(1.00)
Economics Inflation: No terms met the min_df/max_df criteria!
Public Health: health(1.00)


### Step 6: Verify Their Work

Let's install `pytest` and run the repository test suite to verify everything behaves correctly.

In [33]:
!pip install -q pytest
import os
# Ensure repository exists before running pytest
if not os.path.exists('unstructuredata_tfidf_repo'):
    !git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git

!pytest unstructuredata_tfidf_repo -q

...                                                                      [100%]
3 passed in 1.36s
